# 7.13 RNNs, LSTMs, and GRUs

[Back to neural networks index](../7_neural_networks.ipynb)

This notebook compares the classical recurrent architectures used before transformers became dominant, and clarifies where gated recurrence still matters pedagogically.


## 7.13.1 Simple RNN limits

### Why It Matters
A plain RNN can summarize sequences, but it struggles when important information must survive many steps. The parameter count is small, but the memory mechanism is fragile.


In [ ]:
import torch
from torch import nn

rnn = nn.RNN(input_size=8, hidden_size=12, batch_first=True)
x = torch.randn(4, 15, 8)
out, hidden = rnn(x)
{"output_shape": list(out.shape), "final_hidden_shape": list(hidden.shape), "parameter_count": sum(p.numel() for p in rnn.parameters())}


### Interpretation
A simple RNN is valuable to understand, but its memory dynamics are often too weak for long-range dependencies.


## 7.13.2 Gating intuition

### Why It Matters
LSTMs and GRUs add gates that decide what to keep, update, or forget. Even without deriving the equations, parameter counts show that the models are carrying more internal control structure.


In [ ]:
import torch
from torch import nn

rnn = nn.RNN(5, 7, batch_first=True)
lstm = nn.LSTM(5, 7, batch_first=True)
gru = nn.GRU(5, 7, batch_first=True)
{
    "rnn_params": sum(p.numel() for p in rnn.parameters()),
    "lstm_params": sum(p.numel() for p in lstm.parameters()),
    "gru_params": sum(p.numel() for p in gru.parameters()),
}


### Interpretation
The extra parameters correspond to the gating machinery that helps manage memory over time.


## 7.13.3 LSTM versus GRU tradeoffs

### Why It Matters
GRUs are a bit simpler, while LSTMs expose separate cell and hidden states. This example compares their output signatures.


In [ ]:
import torch
from torch import nn

x = torch.randn(3, 6, 4)
lstm = nn.LSTM(4, 5, batch_first=True)
gru = nn.GRU(4, 5, batch_first=True)
lstm_out, (lstm_h, lstm_c) = lstm(x)
gru_out, gru_h = gru(x)
{
    "lstm_output_shape": list(lstm_out.shape),
    "lstm_cell_shape": list(lstm_c.shape),
    "gru_output_shape": list(gru_out.shape),
    "gru_hidden_shape": list(gru_h.shape),
}


### Interpretation
Both are sequence models, but their state objects and internal control differ.


## 7.13.4 Sequence classification

### Why It Matters
A common recurrent use case is classifying an entire sequence from its final state.


In [ ]:
import torch
from torch import nn

torch.manual_seed(36)
X = torch.randint(0, 15, (120, 6))
y = (X.sum(dim=1) > 40).long()
model = nn.Sequential(nn.Embedding(15, 8))
lstm = nn.LSTM(8, 12, batch_first=True)
head = nn.Linear(12, 2)
params = list(model.parameters()) + list(lstm.parameters()) + list(head.parameters())
opt = torch.optim.Adam(params, lr=0.03)
loss_fn = nn.CrossEntropyLoss()
for _ in range(30):
    embeds = model(X)
    _, (hidden, _) = lstm(embeds)
    logits = head(hidden[-1])
    loss = loss_fn(logits, y)
    opt.zero_grad()
    loss.backward()
    opt.step()
with torch.no_grad():
    preds = logits.argmax(dim=1)
    acc = (preds == y).float().mean().item()
{"training_accuracy": round(float(acc), 3), "final_loss": round(float(loss.item()), 4)}


### Interpretation
The final recurrent state is acting as a compressed summary of the full input sequence.


## 7.13.5 Sequence forecasting

### Why It Matters
Recurrent models can also predict the next numeric value in a sequence. A short sine-wave forecasting setup is enough to illustrate the pattern.


In [ ]:
import torch
from torch import nn

torch.manual_seed(37)
series = torch.sin(torch.linspace(0, 20, 220))
windows = torch.stack([series[i:i+10] for i in range(180)]).unsqueeze(-1)
targets = torch.stack([series[i+10] for i in range(180)]).unsqueeze(1)
lstm = nn.LSTM(1, 12, batch_first=True)
head = nn.Linear(12, 1)
opt = torch.optim.Adam(list(lstm.parameters()) + list(head.parameters()), lr=0.03)
loss_fn = nn.MSELoss()
for _ in range(40):
    out, (hidden, _) = lstm(windows)
    preds = head(hidden[-1])
    loss = loss_fn(preds, targets)
    opt.zero_grad()
    loss.backward()
    opt.step()
{"forecast_mse": round(float(loss.item()), 4)}


### Interpretation
The model is using the recent history window to infer the next point in the pattern.


## 7.13.6 Build a small LSTM example

### Why It Matters
This final subsection keeps the setup minimal and trains a tiny LSTM language-style model on character sequences.


In [ ]:
import torch
from torch import nn

text = "neuralnets"
chars = sorted(set(text))
stoi = {ch: i for i, ch in enumerate(chars)}
seqs = []
targets = []
for i in range(len(text) - 3):
    seqs.append([stoi[c] for c in text[i:i+3]])
    targets.append(stoi[text[i+3]])
X = torch.tensor(seqs, dtype=torch.long)
y = torch.tensor(targets, dtype=torch.long)
embed = nn.Embedding(len(chars), 8)
lstm = nn.LSTM(8, 12, batch_first=True)
head = nn.Linear(12, len(chars))
opt = torch.optim.Adam(list(embed.parameters()) + list(lstm.parameters()) + list(head.parameters()), lr=0.05)
loss_fn = nn.CrossEntropyLoss()
for _ in range(80):
    emb = embed(X)
    _, (hidden, _) = lstm(emb)
    logits = head(hidden[-1])
    loss = loss_fn(logits, y)
    opt.zero_grad()
    loss.backward()
    opt.step()
{"final_loss": round(float(loss.item()), 4), "vocab_size": len(chars)}


### Interpretation
The dataset is tiny, but the workflow already looks like a compact language-model training loop.
